# VQA Training on Google Colab
## Visual Question Answering với PyTorch

Notebook này chứa toàn bộ code để train VQA model trên Google Colab.

### Architecture:
- **EncoderCNN**: ResNet-50 để extract visual features
- **DecoderRNN**: LSTM để encode questions và predict answers
- **Answer Prediction**: Classification trên top-K answers phổ biến nhất

## 1. Setup Environment

In [ ]:
# Install dependencies (uncomment nếu cần)
# !pip install torch torchvision tqdm pillow h5py nltk matplotlib pycocotools pyyaml

import os
import time
import json
import numpy as np
from collections import Counter
from typing import Dict, List, Tuple, Optional

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.models as models
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive (nếu cần)

In [ ]:
# Uncomment để mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Đường dẫn đến data
DATA_DIR = 'data'
CHECKPOINT_DIR = 'checkpoints'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 3. Vocabulary Class

In [ ]:
class Vocabulary:
    """
    Vocabulary class for managing word-to-index and index-to-word mappings.
    """
    
    def __init__(self, freq_threshold: int = 5):
        """Initialize Vocabulary."""
        self.freq_threshold = freq_threshold
        self.word2idx = {"<PAD>": 0, "<START>": 1, "<END>": 2, "<UNK>": 3}
        self.idx2word = {0: "<PAD>", 1: "<START>", 2: "<END>", 3: "<UNK>"}
        self.word_freq = {}
        self.idx = 4
        
    def __len__(self) -> int:
        """Return vocabulary size."""
        return len(self.word2idx)
    
    def build_vocabulary(self, sentences: List[str]) -> None:
        """Build vocabulary from a list of sentences."""
        # Count word frequencies
        for sentence in sentences:
            for word in self.tokenize(sentence):
                if word not in self.word_freq:
                    self.word_freq[word] = 0
                self.word_freq[word] += 1
        
        # Add words that meet frequency threshold
        for word, freq in self.word_freq.items():
            if freq >= self.freq_threshold:
                self.word2idx[word] = self.idx
                self.idx2word[self.idx] = word
                self.idx += 1
    
    def tokenize(self, text: str) -> List[str]:
        """Tokenize text into words."""
        return text.lower().strip().split()
    
    def encode(self, text: str, max_length: Optional[int] = None) -> List[int]:
        """Encode text to indices."""
        tokens = self.tokenize(text)
        encoded = [self.word2idx.get(token, self.word2idx["<UNK>"]) for token in tokens]
        
        if max_length:
            if len(encoded) < max_length:
                encoded += [self.word2idx["<PAD>"]] * (max_length - len(encoded))
            else:
                encoded = encoded[:max_length]
        
        return encoded
    
    def decode(self, indices: List[int]) -> str:
        """Decode indices back to text."""
        words = [self.idx2word.get(idx, "<UNK>") for idx in indices]
        words = [w for w in words if w not in ["<PAD>", "<START>", "<END>"]]
        return " ".join(words)

print("Vocabulary class defined!")

## 4. VQA Dataset Class

In [ ]:
class VQADataset(Dataset):
    """
    PyTorch Dataset for Visual Question Answering.
    """
    
    def __init__(
        self,
        image_dir: str,
        questions_file: str,
        annotations_file: Optional[str] = None,
        vocab: Optional[Vocabulary] = None,
        transform: Optional[callable] = None,
        max_question_length: int = 20,
        answer_to_idx: Optional[Dict[str, int]] = None,
        split: str = "train",
    ):
        """Initialize VQA Dataset."""
        self.image_dir = image_dir
        self.transform = transform
        self.vocab = vocab
        self.max_question_length = max_question_length
        self.answer_to_idx = answer_to_idx
        self.split = split
        
        # Load questions
        with open(questions_file, 'r') as f:
            questions_data = json.load(f)
            self.questions = questions_data.get('questions', [])
        
        # Load annotations if provided
        self.annotations = None
        if annotations_file:
            with open(annotations_file, 'r') as f:
                annotations_data = json.load(f)
                self.annotations = {
                    ann['question_id']: ann 
                    for ann in annotations_data.get('annotations', [])
                }
    
    def __len__(self) -> int:
        """Return dataset size."""
        return len(self.questions)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        """Get a single sample from the dataset."""
        question_data = self.questions[idx]
        question_id = question_data['question_id']
        image_id = question_data['image_id']
        question_text = question_data['question']
        
        # Load and preprocess image
        image_filename = f"COCO_{self.split}2014_{image_id:012d}.jpg"
        image_path = f"{self.image_dir}/{image_filename}"
        
        try:
            image = Image.open(image_path).convert('RGB')
        except:
            # Create dummy image if file not found
            image = Image.new('RGB', (224, 224), color='black')
            
        if self.transform:
            image = self.transform(image)
        
        # Encode question
        if self.vocab:
            question_encoded = torch.tensor(
                self.vocab.encode(question_text, self.max_question_length),
                dtype=torch.long
            )
        else:
            question_encoded = torch.tensor([])
        
        # Prepare output
        sample = {
            'image': image,
            'question': question_encoded,
            'question_id': question_id,
        }
        
        # Add answer if annotations available
        if self.annotations and question_id in self.annotations:
            annotation = self.annotations[question_id]
            answers = [ans['answer'] for ans in annotation['answers']]
            answer_text = answers[0] if answers else ""
            
            if self.answer_to_idx and answer_text in self.answer_to_idx:
                answer_idx = self.answer_to_idx[answer_text]
                sample['answer'] = torch.tensor(answer_idx, dtype=torch.long)
        
        return sample
    
    @staticmethod
    def collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
        """Custom collate function for batching."""
        images = torch.stack([item['image'] for item in batch])
        questions = torch.stack([item['question'] for item in batch])
        question_ids = [item['question_id'] for item in batch]
        
        batched = {
            'images': images,
            'questions': questions,
            'question_ids': question_ids,
        }
        
        if 'answer' in batch[0]:
            answers = torch.stack([item['answer'] for item in batch])
            batched['answers'] = answers
        
        return batched


def build_answer_vocab(annotations_file: str, top_k: int = 1000) -> Tuple[Dict[str, int], Dict[int, str]]:
    """Build answer vocabulary from top-k most frequent answers."""
    with open(annotations_file, 'r') as f:
        annotations_data = json.load(f)
        annotations = annotations_data.get('annotations', [])
    
    # Count answer frequencies
    answer_freq = {}
    for ann in annotations:
        for answer_data in ann['answers']:
            answer = answer_data['answer']
            answer_freq[answer] = answer_freq.get(answer, 0) + 1
    
    # Get top-k answers
    top_answers = sorted(answer_freq.items(), key=lambda x: x[1], reverse=True)[:top_k]
    
    # Create mappings
    answer_to_idx = {answer: idx for idx, (answer, _) in enumerate(top_answers)}
    idx_to_answer = {idx: answer for answer, idx in answer_to_idx.items()}
    
    return answer_to_idx, idx_to_answer

print("Dataset class defined!")

## 5. Model Architecture

In [ ]:
class EncoderCNN(nn.Module):
    """CNN-based image encoder using pretrained ResNet."""
    
    def __init__(self, embed_size: int = 512, pretrained: bool = True):
        """Initialize EncoderCNN."""
        super(EncoderCNN, self).__init__()
        
        # Load pretrained ResNet-50
        if pretrained:
            try:
                weights = models.ResNet50_Weights.IMAGENET1K_V1
                resnet = models.resnet50(weights=weights)
            except AttributeError:
                resnet = models.resnet50(pretrained=True)
        else:
            resnet = models.resnet50(weights=None)
        
        # Remove the final fully connected layer
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)
        
        # Linear layer to project ResNet features to embed_size
        self.fc = nn.Linear(resnet.fc.in_features, embed_size)
        self.bn = nn.BatchNorm1d(embed_size)
        
        # Freeze ResNet parameters for initial training
        self.freeze_resnet()
    
    def freeze_resnet(self):
        """Freeze ResNet parameters."""
        for param in self.resnet.parameters():
            param.requires_grad = False
    
    def unfreeze_resnet(self):
        """Unfreeze ResNet parameters for fine-tuning."""
        for param in self.resnet.parameters():
            param.requires_grad = True
    
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """Forward pass. Input: (batch_size, 3, 224, 224) -> Output: (batch_size, embed_size)"""
        features = self.resnet(images)
        features = features.view(features.size(0), -1)
        features = self.fc(features)
        features = self.bn(features)
        return features


class DecoderRNN(nn.Module):
    """LSTM-based decoder for processing questions and generating answers."""
    
    def __init__(
        self,
        embed_size: int,
        hidden_size: int,
        vocab_size: int,
        num_classes: int,
        num_layers: int = 1,
        dropout: float = 0.5,
    ):
        """Initialize DecoderRNN."""
        super(DecoderRNN, self).__init__()
        
        self.embed_size = embed_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Word embedding for questions (padding_idx=0 assumes PAD token is at index 0)
        self.word_embed = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        
        # LSTM for question encoding
        self.lstm = nn.LSTM(
            embed_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        
        self.dropout = nn.Dropout(dropout)
        
        # Fusion layer: combine image features and question features
        self.fusion = nn.Sequential(
            nn.Linear(embed_size + hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        
        # Classifier: predict answer from fused features
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_classes),
        )
    
    def forward(
        self,
        image_features: torch.Tensor,
        questions: torch.Tensor,
    ) -> torch.Tensor:
        """Forward pass. Output: (batch_size, num_classes)"""
        # Embed questions
        question_embeds = self.word_embed(questions)
        question_embeds = self.dropout(question_embeds)
        
        # Encode questions with LSTM
        lstm_out, (hidden, cell) = self.lstm(question_embeds)
        question_features = hidden[-1]
        
        # Fuse image and question features
        combined = torch.cat([image_features, question_features], dim=1)
        fused = self.fusion(combined)
        
        # Classify to get answer logits
        logits = self.classifier(fused)
        
        return logits


class VQAModel(nn.Module):
    """Complete VQA model combining EncoderCNN and DecoderRNN."""
    
    def __init__(
        self,
        embed_size: int = 512,
        hidden_size: int = 512,
        vocab_size: int = 10000,
        num_classes: int = 1000,
        num_layers: int = 1,
        dropout: float = 0.5,
        pretrained: bool = True,
    ):
        """Initialize VQA model."""
        super(VQAModel, self).__init__()
        
        self.encoder = EncoderCNN(embed_size, pretrained)
        self.decoder = DecoderRNN(
            embed_size,
            hidden_size,
            vocab_size,
            num_classes,
            num_layers,
            dropout,
        )
    
    def forward(
        self,
        images: torch.Tensor,
        questions: torch.Tensor,
    ) -> torch.Tensor:
        """Forward pass. Output: (batch_size, num_classes)"""
        image_features = self.encoder(images)
        logits = self.decoder(image_features, questions)
        return logits

print("Model architecture defined!")

## 6. Training Configuration

In [ ]:
# Hyperparameters
EMBED_SIZE = 512
HIDDEN_SIZE = 512
NUM_LAYERS = 1
DROPOUT = 0.5
BATCH_SIZE = 32  # Giảm xuống nếu bị OOM
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
TOP_K_ANSWERS = 1000
MAX_QUESTION_LENGTH = 20

# Paths
TRAIN_IMAGE_DIR = f"{DATA_DIR}/train2014"
TRAIN_QUESTIONS = f"{DATA_DIR}/subset_questions.json"
TRAIN_ANNOTATIONS = f"{DATA_DIR}/subset_annotations.json"

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Image transforms
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

print("Configuration set!")

## 7. Build Vocabularies

In [ ]:
# Build question vocabulary
print("Building question vocabulary...")
question_vocab = Vocabulary(freq_threshold=2)

# Load questions to build vocab
with open(TRAIN_QUESTIONS, 'r') as f:
    questions_data = json.load(f)
    questions = [q['question'] for q in questions_data['questions']]

question_vocab.build_vocabulary(questions)
print(f"Question vocabulary size: {len(question_vocab)}")

# Build answer vocabulary
print("\nBuilding answer vocabulary...")
answer_to_idx, idx_to_answer = build_answer_vocab(TRAIN_ANNOTATIONS, top_k=TOP_K_ANSWERS)
print(f"Answer vocabulary size: {len(answer_to_idx)}")
print(f"\nTop 10 answers: {list(answer_to_idx.keys())[:10]}")

## 8. Create Dataset and DataLoader

In [ ]:
# Create dataset
print("Creating dataset...")
train_dataset = VQADataset(
    image_dir=TRAIN_IMAGE_DIR,
    questions_file=TRAIN_QUESTIONS,
    annotations_file=TRAIN_ANNOTATIONS,
    vocab=question_vocab,
    transform=train_transform,
    max_question_length=MAX_QUESTION_LENGTH,
    answer_to_idx=answer_to_idx,
    split='train'
)

print(f"Dataset size: {len(train_dataset)}")

# Create dataloader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=VQADataset.collate_fn,
)

print(f"Number of batches: {len(train_loader)}")

## 9. Initialize Model, Loss, and Optimizer

In [ ]:
# Create model
print("Creating model...")
model = VQAModel(
    embed_size=EMBED_SIZE,
    hidden_size=HIDDEN_SIZE,
    vocab_size=len(question_vocab),
    num_classes=TOP_K_ANSWERS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pretrained=True,
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

print("Model ready for training!")

## 10. Training Loop

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device, epoch):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]")
    
    for batch_idx, batch in enumerate(pbar):
        # Move data to device
        images = batch['images'].to(device)
        questions = batch['questions'].to(device)
        answers = batch['answers'].to(device)
        
        # Forward pass
        logits = model(images, questions)
        loss = criterion(logits, answers)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        # Update metrics
        total_loss += loss.item()
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == answers).sum().item()
        total += answers.size(0)
        
        # Update progress bar
        avg_loss = total_loss / (batch_idx + 1)
        accuracy = 100.0 * correct / total
        pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'acc': f'{accuracy:.2f}%'})
    
    return avg_loss, accuracy

print("Training function defined!")

## 11. Start Training

In [ ]:
print(f"Starting training for {NUM_EPOCHS} epochs...")
print(f"Training samples: {len(train_dataset)}")
print(f"Batches per epoch: {len(train_loader)}\n")

# Training history
history = {
    'train_loss': [],
    'train_acc': []
}

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start_time = time.time()
    
    # Train
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, device, epoch
    )
    
    # Update scheduler
    scheduler.step()
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    
    # Save checkpoint
    checkpoint_path = f"{CHECKPOINT_DIR}/checkpoint_epoch_{epoch}.pth"
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'train_acc': train_acc,
    }, checkpoint_path)
    
    epoch_time = time.time() - epoch_start_time
    print(f"\nEpoch {epoch}/{NUM_EPOCHS} - "
          f"Time: {epoch_time:.2f}s - "
          f"Loss: {train_loss:.4f} - "
          f"Acc: {train_acc:.2f}%\n")

print("Training completed!")

## 12. Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot loss
ax1.plot(history['train_loss'], marker='o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True)

# Plot accuracy
ax2.plot(history['train_acc'], marker='o', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training Accuracy')
ax2.grid(True)

plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/training_history.png")
plt.show()

print("Training history plotted!")

## 13. Inference Example

In [ ]:
def predict_answer(model, image_path, question, vocab, answer_idx_to_text, transform, device):
    """Predict answer for a question about an image."""
    model.eval()
    
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    # Encode question
    question_encoded = torch.tensor(
        vocab.encode(question, MAX_QUESTION_LENGTH),
        dtype=torch.long
    ).unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        logits = model(image_tensor, question_encoded)
        predicted_idx = torch.argmax(logits, dim=1).item()
    
    predicted_answer = answer_idx_to_text.get(predicted_idx, "<UNK>")
    
    return predicted_answer

# Example usage (uncomment khi đã có ảnh)
# image_path = f"{TRAIN_IMAGE_DIR}/COCO_train2014_000000000009.jpg"
# question = "What is in the image?"
# answer = predict_answer(model, image_path, question, question_vocab, idx_to_answer, train_transform, device)
# print(f"Question: {question}")
# print(f"Predicted Answer: {answer}")

print("Inference function defined!")

## Hoàn tất!

Bạn đã train xong VQA model trên Google Colab. 

### Các file được tạo:
- Checkpoints trong thư mục `checkpoints/`
- Training history plot: `checkpoints/training_history.png`

### Để sử dụng model:
1. Load checkpoint
2. Sử dụng function `predict_answer()` để dự đoán

### Tips cho Colab:
- Nếu bị OOM (Out of Memory), giảm `BATCH_SIZE`
- Có thể lưu checkpoints vào Google Drive để không mất khi session kết thúc
- Sử dụng GPU runtime để train nhanh hơn